# 3.1 — ML models trên folds chung

Chạy lại đúng 5 model của ML task 1.5 (`1.ML/src/task_1_5_decimal_fe.py`): LightGBM `nl3`/`nl4`/`nl8`, Spline LR, XGBoost depth 3,
cùng feature (decimal fractions + interactions), nhưng đọc fold từ `folds_5_seed9.csv`.
Mỗi model ghi riêng `3.ENSEMBLE/oof/<model>.csv` (xác suất OOF) và `3.ENSEMBLE/test/<model>.csv` (trung bình xác suất 5 fold).

**Input:** competition + `ccf-shared-code` + output notebook 3.0. **Accelerator:** None (CPU).
**Kiểm tra:** rank blend OOF ≈ 0.94792 như ML task 1.5.

In [ ]:
import sys
from pathlib import Path

# shared/ code: local repo or the Kaggle dataset that holds features.py + dl.py
cands = [Path("shared/features.py"), Path("../../shared/features.py"), *Path("/kaggle/input").rglob("features.py")]
feat = next(p for p in cands if p.is_file())
sys.path.insert(0, str(feat.parent.resolve()))
import features as F
print("shared code:", feat.parent)

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

tr, te, y, train_id, test_id = F.load()
fold = F.load_folds(train_id)          # from notebook 3.0 output (folds_5_seed9.csv)
assert len(tr) == F.N_TRAIN and len(te) == F.N_TEST and set(np.unique(fold)) == set(range(F.N_FOLDS))
log(f"train {tr.shape}, test {te.shape}, fraud rate {y.mean():.4%}")

In [ ]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LogisticRegression

# Có thể chạy từng phần nếu sợ quá giờ, vd. MODELS = ["xgb"]
MODELS = ["lgb_nl3", "lgb_nl4", "lgb_nl8", "spline_lr", "xgb"]
SEED = F.SEED

LGB_BASE = dict(objective="binary", metric="auc", bagging_freq=1, verbose=-1, n_jobs=-1,
                min_data_per_group=200, cat_l2=10)
LGB_CONFIGS = {
    "lgb_nl3": dict(learning_rate=0.06, num_leaves=3, min_child_samples=200, feature_fraction=0.8,
                    bagging_fraction=0.8, lambda_l2=10, cat_smooth=20, seed=SEED),
    "lgb_nl4": dict(learning_rate=0.06, num_leaves=4, min_child_samples=200, feature_fraction=0.7,
                    bagging_fraction=0.8, lambda_l2=10, cat_smooth=20, seed=SEED + 1),
    "lgb_nl8": dict(learning_rate=0.04, num_leaves=8, min_child_samples=300, feature_fraction=0.6,
                    bagging_fraction=0.7, lambda_l2=30, lambda_l1=1, cat_smooth=30, seed=SEED + 2),
}
XGB_PARAMS = dict(objective="binary:logistic", eval_metric="auc", eta=0.06, max_depth=3, min_child_weight=20,
                  subsample=0.8, colsample_bytree=0.7, reg_lambda=10, tree_method="hist", nthread=-1, seed=SEED)


def fit_lgb(name, Xa, ya, Xb, yb, Xt):
    m = lgb.train({**LGB_BASE, **LGB_CONFIGS[name]}, lgb.Dataset(Xa, ya), 6000, valid_sets=[lgb.Dataset(Xb, yb)],
                  callbacks=[lgb.early_stopping(150, verbose=False)])
    it = m.best_iteration
    return m.predict(Xb, num_iteration=it), m.predict(Xt, num_iteration=it), it


def fit_linear(Aa, ya, Ab, yb, At):
    m = LogisticRegression(C=0.1, max_iter=400).fit(Aa, ya)
    return m.predict_proba(Ab)[:, 1], m.predict_proba(At)[:, 1], int(m.n_iter_[0])


def fit_xgb(Xa, ya, Xb, yb, Xt):
    da, db, dt = xgb.DMatrix(Xa, ya), xgb.DMatrix(Xb, yb), xgb.DMatrix(Xt)
    m = xgb.train(XGB_PARAMS, da, 6000, evals=[(db, "v")], early_stopping_rounds=150, verbose_eval=False)
    r = (0, m.best_iteration + 1)
    return m.predict(db, iteration_range=r), m.predict(dt, iteration_range=r), m.best_iteration + 1

In [ ]:
# Feature matrices giống ML task 1.5: tree (category dtype), codes (XGB), linear (spline + one-hot, fit trên train+test)
Xtr_t, Xte_t = F.tree_frame(tr), F.tree_frame(te)
Xtr_c, Xte_c = F.codes_frame(tr), F.codes_frame(te)
lin = F.LinearFE().fit(pd.concat([tr, te], ignore_index=True))
Atr, Ate = lin.transform(tr), lin.transform(te)
log(f"tree features: {Xtr_t.shape[1]}, linear features: {Atr.shape[1]}")

In [ ]:
all_metrics = {}
for name in MODELS:
    oof = np.zeros(len(tr))
    test_pred = np.zeros(len(te))
    iters = []
    for f, a, b in F.iter_folds(fold):
        t0 = time.time()
        if name in LGB_CONFIGS:
            o, p, it = fit_lgb(name, Xtr_t.iloc[a], y[a], Xtr_t.iloc[b], y[b], Xte_t)
        elif name == "spline_lr":
            o, p, it = fit_linear(Atr[a], y[a], Atr[b], y[b], Ate)
        else:
            o, p, it = fit_xgb(Xtr_c.iloc[a], y[a], Xtr_c.iloc[b], y[b], Xte_c)
        oof[b] = o
        test_pred += p / F.N_FOLDS
        iters.append(int(it))
        log(f"{name:9s} fold {f} | AUC={roc_auc_score(y[b], o):.5f} | iters={it} | {time.time() - t0:.0f}s")
    all_metrics[name] = F.save_predictions(name, train_id, oof, test_id, test_pred, y, fold,
                                           extra={"source": "3.1 (ML task 1.5 config)", "iterations": iters})
    m = all_metrics[name]
    log(f"==> {name}: OOF AUC={m['oof_roc_auc']:.5f} | AP={m['oof_average_precision']:.5f} "
        f"| fold std={m['fold_roc_auc_std']:.5f}")

In [ ]:
# Kiểm tra tái lập: rank blend như ML task 1.5 (tuned + shrink 0.7/0.3) -> kỳ vọng ≈ 0.94792
names = [m for m in ["lgb_nl3", "lgb_nl4", "lgb_nl8", "spline_lr", "xgb"] if m in MODELS]
O, _ = F.load_predictions(names, train_id, test_id, roots=("3.ENSEMBLE",))
R = np.column_stack([F.rank01(O[:, i]) for i in range(O.shape[1])])
eq = roc_auc_score(y, R.mean(1))
w, tuned = F.tune_weights(R, y)
w = 0.7 * w + 0.3 / len(names)
shrunk = roc_auc_score(y, R @ w)

summary = pd.DataFrame(all_metrics).T[["oof_roc_auc", "oof_average_precision", "fold_roc_auc_mean", "fold_roc_auc_std"]]
print(summary.astype(float).round(5))
print(f"\nEqual blend OOF AUC  = {eq:.5f}")
print(f"Tuned blend OOF AUC  = {tuned:.5f}")
print(f"Shrunk blend OOF AUC = {shrunk:.5f}   (ML task 1.5 báo cáo: 0.94792)")
print("Weights:", dict(zip(names, np.round(w, 3))))

summary.to_csv("3.ENSEMBLE/results/ml_models_summary.csv")
Path("3.ENSEMBLE/results/ml_blend_check.json").write_text(json.dumps(
    {"models": names, "equal": eq, "tuned": tuned, "shrunk": shrunk, "weights": dict(zip(names, w.tolist()))}, indent=2))
log("done")